# Predicting Airline Passenger Satisfaction with Kumo-Tabular

*Playground Series — Season 6, Episode 10*

**[Kumo-Tabular](https://huggingface.co/nvidia/Kumo-Tabular)** is NVIDIA's pretrained foundation model for tabular data. Give it labeled examples as **context**, then ask it to predict new rows. There is no model training loop or feature engineering here: we use its pretrained weights and built-in preprocessing.

This starter uses **one GPU**, the **small** model, **100,000 context rows**, **2 estimators**, and **fp16**. A single five-fold loop produces out-of-fold (OOF) predictions and averaged test probabilities. To improve CV and LB score, we can use larger context and/or more estimators. We can also rewrite the code to use both 2xT4 GPUs for 2x speedup.

## Setup

Following instructions on NVIDIA Kumo-Tabular's Hugging Face [here][1], we install [structured-data-models][2] for inference. The model downloads its pretrained weights when created.

[1]: https://huggingface.co/nvidia/Kumo-Tabular
[2]: https://github.com/NVIDIA/structured-data-models

In [1]:
%pip install -q "git+https://github.com/NVIDIA/structured-data-models.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import time
import numpy as np
import pandas as pd
import torch
import sdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

DATA_DIR = "/kaggle/input/competitions/playground-series-s6e10"
TARGET = "satisfaction"
CONTEXT_ROWS = 100_000 # use None for full context
N_ESTIMATORS = 2 # Kumo recommends 8

assert torch.cuda.is_available(), "Enable a GPU in Kaggle settings."
device = torch.device("cuda:0")
torch.cuda.set_device(device)
print(torch.cuda.get_device_name(device))

Tesla T4


## Load the data

Use the raw features, excluding `id`. Categorical columns remain categorical and missing numerical values stay missing. The target is represented as categorical strings `"0"` and `"1"`, so we can select the probability of class `"1"` by name.

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
FEATURES = [c for c in test.columns if c != "id"]
y = train[TARGET].astype(int).to_numpy()
assert set(np.unique(y)) == {0, 1}

stypes = sdm.infer_stypes(
    train[FEATURES + [TARGET]],
    overrides={TARGET: "categorical"},
)
print(f"Train: {train.shape} | Test: {test.shape}")
train.head()

Train: (699635, 23) | Test: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


## Predict with five-fold cross-validation

Each fold samples **100,000 context rows** from its training portion, preserving the class balance. Held-out labels are never passed to the model.

The call follows the [Hugging Face example](https://huggingface.co/nvidia/Kumo-Tabular): `x_context`, `y_context`, and `x_query` go in; class probabilities come out. We explicitly select `size="small"`.

Validation rows and test rows are combined into one query per fold. The first predictions go into `oof`; the rest contribute to `test_preds`.

In [4]:
model = sdm.models.KumoTabular(task="classification", size="small", device=device)

oof = np.zeros(len(train), dtype=np.float64)
test_preds = np.zeros(len(test), dtype=np.float64)
skf = StratifiedKFold(5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(train, y)):
    start = time.time()
    torch.manual_seed(42 + fold)
    context_idx = train_idx
    if CONTEXT_ROWS is not None and len(context_idx) > CONTEXT_ROWS:
        context_idx, _ = train_test_split(
            context_idx, train_size=CONTEXT_ROWS,
            stratify=y[context_idx], random_state=42,
        )

    # Context first, then validation and test queries, all with one shared schema.
    context = train.iloc[context_idx][FEATURES].copy()
    context[TARGET] = y[context_idx].astype(str)
    query = pd.concat([train.iloc[valid_idx][FEATURES], test[FEATURES]], ignore_index=True)
    query[TARGET] = "0"  # Placeholder only: query targets are dropped below.
    df = pd.concat([context, query], ignore_index=True)
    table = sdm.TableTensor.from_pandas(df=df, stypes=stypes, device=device)
    n_context = len(context_idx)
    print(f"Fold {fold + 1}: {n_context:,} context rows, {len(query):,} query rows", flush=True)

    with torch.amp.autocast(device.type, dtype=torch.float16):
        probs = model(
            x_context=table[:n_context].drop_columns(TARGET),
            y_context=table[:n_context, TARGET],
            x_query=table[n_context:].drop_columns(TARGET),
            num_estimators=N_ESTIMATORS,
        )

    # Kumo returns a TableTensor with one numerical column per class.
    columns = probs.columns
    class_names = list(columns[sdm.Stype.numerical] if isinstance(columns, dict) else columns)
    pred = probs.numerical[:, class_names.index("1")].float().cpu().numpy()
    assert pred.shape == (len(valid_idx) + len(test),) and np.isfinite(pred).all()
    oof[valid_idx] = pred[:len(valid_idx)]
    test_preds += pred[len(valid_idx):] / skf.n_splits

    auc = roc_auc_score(y[valid_idx], oof[valid_idx])
    print(f"Fold {fold + 1} AUC: {auc:.6f} | {(time.time() - start) / 60:.1f} min", flush=True)
    del table, probs, pred, df, context, query
    gc.collect()
    torch.cuda.empty_cache()

print(f"Overall OOF AUC: {roc_auc_score(y, oof):.6f}")

config.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

small/classifier.pt: reconstructing file:   0%|          |  0.00B /  110MB            

small/classifier.pt: downloading bytes:           |  0.00B            

Fold 1: 100,000 context rows, 439,771 query rows
Fold 1 AUC: 0.957585 | 4.5 min
Fold 2: 100,000 context rows, 439,771 query rows
Fold 2 AUC: 0.956618 | 4.4 min
Fold 3: 100,000 context rows, 439,771 query rows
Fold 3 AUC: 0.958120 | 4.4 min
Fold 4: 100,000 context rows, 439,771 query rows
Fold 4 AUC: 0.957664 | 4.4 min
Fold 5: 100,000 context rows, 439,771 query rows
Fold 5 AUC: 0.957501 | 4.4 min
Overall OOF AUC: 0.957435


## Save predictions and create the submission

Each training row has one held-out prediction. Test probabilities are averaged over all five folds. Save both arrays for later blending or stacking.

In [5]:
np.save("/kaggle/working/oof_kumo_tabular_small.npy", oof)
np.save("/kaggle/working/test_kumo_tabular_small.npy", test_preds)
pd.DataFrame({"id": train["id"], TARGET: y, "oof_pred": oof}).to_csv(
    "/kaggle/working/oof_kumo_tabular_small.csv", index=False,
)

submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
assert np.array_equal(submission["id"].to_numpy(), test["id"].to_numpy())
submission[TARGET] = test_preds
submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.head()

,id,satisfaction
0,699635,0.183852
1,699636,0.804937
2,699637,0.927082
3,699638,0.041925
4,699639,0.039104
